# Diabetes (Playground S5E12): Conv1D CNN-3 vs CNN-5 in TensorFlow and PyTorch

**Author:** Lưu Anh Dũng (B23DCDK036) · E23CNPM02 · Intelligent System Development · Assignment 05

**Data:** Kaggle competition [Playground Series S5E12, "Diabetes Prediction Challenge"](https://www.kaggle.com/competitions/playground-series-s5e12).
`train.csv` has 700,000 patients × 24 features (age, BMI, blood pressure, cholesterol, lifestyle, demographics) and
the target `diagnosed_diabetes` (0 = no, 1 = yes). A stratified sample of **300,000** rows is used.
Kaggle generated this data synthetically from a real dataset, so results describe the benchmark, not clinical accuracy.

A table has no pixels, so the CNNs here use **1-D convolutions** that slide along the list of features.

| Run ID | Framework | Model |
|---|---|---|
| `DB-TF-CNN-3` | TensorFlow/Keras | 3 Conv1D layers |
| `DB-TF-CNN-5` | TensorFlow/Keras | 5 Conv1D layers |
| `DB-PT-CNN-3` | PyTorch | 3 Conv1D layers |
| `DB-PT-CNN-5` | PyTorch | 5 Conv1D layers |
| `DB-TF-MLP` | TensorFlow/Keras | baseline without convolution (section 6) |

Sections 1–5 explain the method. Results are discussed in sections 6–7.

## 1. Setup

Seed 42 everywhere (same weights, batch order and dropout on every rerun). PyTorch device: `cuda` → `mps` → `cpu`;
TensorFlow finds a GPU by itself or uses the CPU. `A5_SMOKE=1` = quick test run.

In [ ]:
import copy
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


SEED = 42
warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(SEED)

import torch
import torch.nn as nn

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
print("PyTorch", torch.__version__, "| device:", DEVICE)

**Helper functions** (no model code): project paths, the saved split, test metrics, result files and plots.
Settings shared by both frameworks: batch 256, max 20 epochs, early-stopping patience 3, learning rate 0.001.

In [ ]:
import json
import os
import random
from pathlib import Path

from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

SMOKE = os.environ.get("A5_SMOKE") == "1"          # quick test: small subsample, 2 epochs


def find_root():
    """assignment_05/ = first folder above the working directory that has dataset/ and notebook/."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "dataset").is_dir() and (p / "notebook").is_dir():
            return p
    return here.parent


ROOT = find_root()
DATA_DIR = Path(os.environ.get("A5_DATA_DIR", ROOT / "dataset"))
RESULTS_DIR = ROOT / "notebook" / "results" / ("smoke" if SMOKE else "")
MODELS_DIR = ROOT / "notebook" / "models" / ("smoke" if SMOKE else "")
FIG_DIR = ROOT / "report" / "images" / ("smoke" if SMOKE else "")
for d in (RESULTS_DIR, MODELS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
EPOCHS = 2 if SMOKE else 20
PATIENCE = 3
LEARNING_RATE = 1e-3
SMOKE_SIZE = {"train": 2000, "validation": 500, "test": 500}

random.seed(SEED)
np.random.seed(SEED)
print("project folder:", ROOT, "| smoke run:", SMOKE, "| epochs:", EPOCHS, "| batch size:", BATCH_SIZE)


def find_file(code, name):
    """Find `name` anywhere under dataset/<code>/ (unzipping may add a subfolder)."""
    base = DATA_DIR / code.lower()
    hits = sorted(base.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {base}; see dataset/README.md")
    return hits[0]


def make_or_load_splits(code, y, n_sample=None):
    """Stratified 70/15/15 split of row indices, saved once to dataset/<code>/splits.npz."""
    path = DATA_DIR / code.lower() / "splits.npz"
    n_sample = min(n_sample or len(y), len(y))
    splits = None
    if path.exists():
        saved = np.load(path)
        splits = {k: saved[k] for k in ("train", "validation", "test")}
        if sum(len(v) for v in splits.values()) != n_sample:
            splits = None
    if splits is None:
        idx = np.arange(len(y))
        if n_sample < len(y):
            idx, _ = train_test_split(idx, train_size=n_sample, stratify=y, random_state=SEED)
        train, rest = train_test_split(idx, train_size=0.70, stratify=y[idx], random_state=SEED)
        val, test = train_test_split(rest, test_size=0.50, stratify=y[rest], random_state=SEED)
        splits = {"train": np.sort(train), "validation": np.sort(val), "test": np.sort(test)}
        np.savez(path, **splits)
    if SMOKE:
        rng = np.random.default_rng(SEED)
        splits = {k: np.sort(rng.choice(v, min(SMOKE_SIZE[k], len(v)), replace=False)) for k, v in splits.items()}
    return splits


def classification_metrics(y_true, y_pred, y_score=None, binary=False):
    """Accuracy + macro precision/recall/F1; for binary tasks also positive-class F1, ROC-AUC, AP."""
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if binary:
        m["f1_positive"] = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        m["roc_auc"] = roc_auc_score(y_true, y_score)
        m["average_precision"] = average_precision_score(y_true, y_score)
    return {k: float(v) for k, v in m.items()}


class Stopwatch:
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self._start


def save_result(record):
    (RESULTS_DIR / f"{record['run_id']}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")


def load_results(run_ids):
    paths = [RESULTS_DIR / f"{rid}.json" for rid in run_ids]
    return [json.loads(p.read_text(encoding="utf-8")) for p in paths if p.exists()]


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


def plot_history(history, run_id):
    """Figure with 2 plots: loss and accuracy per epoch, train vs validation."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    epochs = np.arange(1, len(history["loss"]) + 1)
    for ax, key, title in [(axes[0], "loss", "Loss"), (axes[1], "accuracy", "Accuracy")]:
        ax.plot(epochs, history[key], marker="o", label="train")
        ax.plot(epochs, history[f"val_{key}"], marker="o", label="validation")
        ax.set_title(f"{run_id}: {title}"); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); savefig(fig, f"{run_id}_curves"); plt.show()


def plot_confusion(y_true, y_pred, labels, run_id):
    """Row-normalized confusion matrix: row = true class, column = predicted class."""
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    shown = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    many = len(labels) > 12
    size = max(4.5, 0.28 * len(labels))
    fig, ax = plt.subplots(figsize=(size + 1, size))
    im = ax.imshow(shown, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90 if many else 0, fontsize=7 if many else 9)
    ax.set_yticklabels(labels, fontsize=7 if many else 9)
    if not many:
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{shown[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if shown[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
    ax.set_title(f"{run_id}: confusion matrix (row-normalized)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); savefig(fig, f"{run_id}_confusion"); plt.show()
    return cm


def results_table(records):
    rows = []
    for r in records:
        row = {"run_id": r["run_id"], "framework": r["framework"], "model": r["model"], "params": r["params"],
               "epochs": r["epochs_run"], "time_per_epoch_s": r["time_per_epoch_s"],
               "train_time_s": r["train_time_s"], "inference_ms_per_1k": r["inference_ms_per_1k"]}
        row.update(r["test_metrics"])
        rows.append(row)
    return pd.DataFrame(rows).set_index("run_id")

In [ ]:
DATASET = "DB"
MODELS = ["CNN-3", "CNN-5"]
FRAMEWORKS = ["TF", "PT"]
RUN_IDS = [f"{DATASET}-{fw}-{m}" for fw in FRAMEWORKS for m in MODELS]
RUN_IDS

BASELINE_ID = f"{DATASET}-TF-MLP"

## 2. Data

### 2.1 Load

| Column group | Count | Examples | Treatment (2.3) |
|---|---|---|---|
| numeric | 15 | age, bmi, systolic_bp, triglycerides, sleep_hours_per_day | standardize |
| categorical (text) | 6 | gender, ethnicity, smoking_status, income_level | one-hot |
| binary 0/1 | 3 | family_history_diabetes, hypertension_history | keep |
| `id` | 1 | row number | drop |
| target | 1 | `diagnosed_diabetes` | `y` |

Only `train.csv` has labels (the competition's `test.csv` does not), so all splits come from it.

In [ ]:
raw = pd.read_csv(find_file(DATASET, "train.csv"))
TARGET = "diagnosed_diabetes"
CATEGORICAL = ["gender", "ethnicity", "education_level", "income_level", "smoking_status", "employment_status"]
BINARY = ["family_history_diabetes", "hypertension_history", "cardiovascular_history"]
NUMERIC = [c for c in raw.columns if c not in CATEGORICAL + BINARY + ["id", TARGET]]

y_raw = raw[TARGET].astype(np.int64).to_numpy()
CLASS_LABELS = ["no diabetes (0)", "diabetes (1)"]
print("rows:", len(raw), "| numeric:", len(NUMERIC), "| categorical:", len(CATEGORICAL), "| binary:", len(BINARY),
      "| missing values:", int(raw.isna().sum().sum()))
raw.head()

### 2.2 Sample 300,000 rows and split 70 / 15 / 15

Both steps are **stratified**: if 60% of all rows are diabetic, the sample and each split are ~60% diabetic too.
train (210k) updates the weights · validation (45k) decides when to stop · test (45k) gives the final score.
Indices are saved in `dataset/db/splits.npz`, so TensorFlow and PyTorch use the same rows.

In [ ]:
splits = make_or_load_splits(DATASET, y_raw, n_sample=300_000)
df_train, df_val, df_test = (raw.iloc[splits[k]] for k in ("train", "validation", "test"))
y_train, y_val, y_test = (y_raw[splits[k]] for k in ("train", "validation", "test"))
pd.DataFrame({"rows": [len(y_train), len(y_val), len(y_test)],
              "diabetes share": [y_train.mean(), y_val.mean(), y_test.mean()]},
             index=["train", "validation", "test"]).round(4)

### 2.3 Turn each row into numbers

- **Standardize numeric columns:** `z = (x − μ) / σ`, with μ and σ from the **training rows only**.
  Example: BMI with μ = 25.9, σ = 2.9 → a BMI of 31.7 becomes `(31.7 − 25.9) / 2.9 = 2.0` ("2 standard deviations above average").
  Without this, triglycerides (~100–300) would outweigh a 0/1 flag just because of its units.
- **One-hot categorical columns:** one 0/1 column per category.
  `smoking_status = "Former"` → `Current 0, Former 1, Never 0`. No fake order (Never < Former < Current) is invented.
- **Binary flags** stay 0/1.
- Using only training rows to compute μ, σ and the category list avoids **data leakage** (test rows must not
  influence the preprocessing).

Result: each patient = a vector of `d` numbers (about 40). For Conv1D it is treated as a sequence of length `d` with
1 channel: Keras `(N, d, 1)`, PyTorch `(N, 1, d)`.

The second cell shows one real training row before and after.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), NUMERIC),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ("binary", "passthrough", BINARY),
])
preprocess.fit(df_train)                                   # mu, sigma, categories: training rows only

X_train, X_val, X_test = (preprocess.transform(d).astype(np.float32) for d in (df_train, df_val, df_test))
FEATURE_NAMES = [n.split("__", 1)[1] for n in preprocess.get_feature_names_out()]
N_FEATURES = X_train.shape[1]

X_train_tf, X_val_tf, X_test_tf = (x[:, :, None] for x in (X_train, X_val, X_test))    # (N, d, 1)
X_train_pt, X_val_pt, X_test_pt = (x[:, None, :] for x in (X_train, X_val, X_test))    # (N, 1, d)
print("d =", N_FEATURES, "features | TF:", X_train_tf.shape, "| PT:", X_train_pt.shape)

In [ ]:
row = 0
before = df_train.iloc[row][["age", "bmi", "triglycerides", "smoking_status", "family_history_diabetes"]]
after = pd.Series(X_train[row], index=FEATURE_NAMES)
after = after[["age", "bmi", "triglycerides"] + [f for f in FEATURE_NAMES if f.startswith("smoking_status")]
              + ["family_history_diabetes"]]
display(before.to_frame("raw value").T)
display(after.round(3).to_frame("model input").T)

### 2.4 Class weights

If one class is much larger, the loss is dominated by it. Each class gets `w_c = N / (2 · N_c)`:
e.g. 62% diabetic / 38% not → `w_1 = 1 / (2·0.62) = 0.81`, `w_0 = 1 / (2·0.38) = 1.32`.
Each row's loss is multiplied by its class weight, so both classes count equally in total.
Used only if the smaller class is below 40%; otherwise all weights are 1. Same weights in both frameworks.

In [ ]:
counts = np.bincount(y_train, minlength=2)
minority_share = counts.min() / counts.sum()
USE_CLASS_WEIGHTS = minority_share < 0.40
class_weight = (counts.sum() / (2 * counts)) if USE_CLASS_WEIGHTS else np.ones(2)
w_train = class_weight[y_train].astype(np.float32)
print("class counts:", counts, "| minority share:", round(minority_share, 3),
      "| weights used:", USE_CLASS_WEIGHTS, "| w_0, w_1 =", class_weight.round(3))

## 3. Look at the data (training rows only)

**Target balance** per split: decides class weights (2.4) and why ROC-AUC and F1 are reported next to accuracy.

In [ ]:
balance = pd.DataFrame({k: np.bincount(y_raw[v], minlength=2) for k, v in splits.items()}, index=CLASS_LABELS)
fig, ax = plt.subplots(figsize=(6, 3.5))
balance.T.plot.bar(ax=ax, rot=0)
ax.set_ylabel("rows"); ax.set_title(f"{DATASET}: target balance per split")
fig.tight_layout(); savefig(fig, f"{DATASET}_target_balance"); plt.show()
balance

**Which numeric features separate the classes?** For each feature: `(mean in diabetic rows − mean in non-diabetic rows) / σ`.
A bar of +0.5 means diabetic patients are on average half a standard deviation higher. The second figure shows the
full distributions of the two strongest features.

In [ ]:
std = df_train[NUMERIC].std()
effect = ((df_train.loc[y_train == 1, NUMERIC].mean() - df_train.loc[y_train == 0, NUMERIC].mean()) / std)
effect = effect.sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
effect.plot.barh(ax=ax, color=np.where(effect > 0, "tab:orange", "tab:blue"))
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("(mean diabetic − mean non-diabetic) / std")
ax.set_title(f"{DATASET}: numeric feature difference between classes")
fig.tight_layout(); savefig(fig, f"{DATASET}_feature_effect"); plt.show()

top2 = effect.abs().sort_values(ascending=False).index[:2]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col in zip(axes, top2):
    for cls, color in [(0, "tab:blue"), (1, "tab:orange")]:
        ax.hist(df_train.loc[y_train == cls, col], bins=50, density=True, alpha=0.5, color=color, label=CLASS_LABELS[cls])
    ax.set_title(col); ax.legend(fontsize=8)
fig.suptitle(f"{DATASET}: the two most separating numeric features (density)")
fig.tight_layout(); savefig(fig, f"{DATASET}_top_features"); plt.show()

**Correlation** `r = cov(a, b) / (σ_a σ_b)`, from −1 to +1, between numeric/binary features and the target.
It only measures straight-line relations; the models can also learn curved ones.

In [ ]:
corr = df_train[NUMERIC + BINARY + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046); ax.set_title(f"{DATASET}: correlation matrix (train)")
fig.tight_layout(); savefig(fig, f"{DATASET}_correlation"); plt.show()

**Categories:** share of diabetic rows in each category, all 6 categorical columns in one chart. Dashed line = overall share.

In [ ]:
rates = []
for col in CATEGORICAL:
    r = pd.Series(y_train, index=df_train.index).groupby(df_train[col]).mean()
    rates += [(f"{col}: {cat}", v) for cat, v in r.items()]
rates = pd.Series(dict(rates))

fig, ax = plt.subplots(figsize=(7, max(4, 0.25 * len(rates))))
rates.iloc[::-1].plot.barh(ax=ax, color="tab:orange")
ax.axvline(y_train.mean(), ls="--", color="black", lw=1)
ax.set_xlabel("share with diabetes"); ax.set_title(f"{DATASET}: diabetes share per category (train)")
fig.tight_layout(); savefig(fig, f"{DATASET}_categorical_rates"); plt.show()

## 4. TensorFlow / Keras

### 4.1 Input pipeline

`tf.data`: shuffle (train only) → batch 256. A training batch is `(features (256, d, 1), label (256,), weight (256,))`;
Keras multiplies each row's loss by its weight.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_tf_dataset(x, y, w=None, training=False):
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((x, y) if w is None else (x, y, w))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_tf_dataset(X_train_tf, y_train, w_train, training=True)
val_ds = make_tf_dataset(X_val_tf, y_val)
test_ds = make_tf_dataset(X_test_tf, y_test)
train_ds.element_spec

### 4.2 How a 1-D convolution reads a row

A Conv1D filter of size 3 slides along the feature vector and mixes **3 neighbouring features**:
`out[i] = w₁·x[i−1] + w₂·x[i] + w₃·x[i+1] + b`.

Example: 6 standardized features `x = [0.5, −1.2, 0.3, 2.0, −0.4, 0.0]`, filter `w = [1, 0, −1]`, `b = 0`,
`padding="same"` (a 0 added at each end, so the output also has 6 values):
- position 1: `1·0.5 + 0·(−1.2) + (−1)·0.3 = 0.2`
- position 3: `1·0.3 + 0·2.0 + (−1)·(−0.4) = 0.7`

Then, as in any conv block: **BatchNorm** `(x − μ)/σ · γ + β` per channel, and **ReLU** `max(0, x)` (`−1.2 → 0`).

**Limitation:** "neighbours" are only neighbours because of the column order from 2.3 (numeric, one-hot, binary).
Unlike pixels, table columns have no natural order. That is why an MLP baseline is added in section 6.
The cell checks the hand result against `tf.nn.conv1d`.

In [ ]:
x = np.array([0.5, -1.2, 0.3, 2.0, -0.4, 0.0], dtype=np.float32)
w = np.array([1, 0, -1], dtype=np.float32)
padded = np.pad(x, 1)                                                     # "same": one 0 at each end
by_hand = np.array([np.sum(padded[i:i + 3] * w) for i in range(len(x))])
tf_out = tf.nn.conv1d(x[None, :, None], w[:, None, None], stride=1, padding="SAME")[0, :, 0].numpy()
print("Conv1D by hand:", by_hand.round(2))
print("same as tf.nn.conv1d:", np.allclose(by_hand, tf_out))
print("after ReLU:    ", np.maximum(0, by_hand).round(2))

### 4.3 The two models

No pooling: the sequence is only `d` ≈ 40 long, and merging neighbours would mix unrelated features.

| Step | CNN-3 output | CNN-5 output |
|---|---|---|
| input | d×1 | d×1 |
| conv blocks (Conv1D k=3 → BatchNorm → ReLU) | 32 → 64 → 64 → **d×64** | 32 → 32 → 64 → 64 → 128 → **d×128** |
| head | Flatten → Dropout 0.3 → Dense 64 → **Dense 1** | same |
| features one output position sees | 7 | 11 |

- **Flatten, not global average pooling.** In a table, position = feature (position 0 is always `age`).
  Averaging `[0.9 at bmi, 0.1 at age]` gives `0.5` and forgets which feature was high; Flatten keeps both values.
- **Dropout 0.3:** 30% of the flattened values set to 0 during training.
- **Dense 1:** one logit `z`; probability of diabetes `p = σ(z) = 1 / (1 + e^(−z))`.

The second cell prints the output shape of every layer for one patient.

In [ ]:
ARCH = {"CNN-3": [32, 64, 64], "CNN-5": [32, 32, 64, 64, 128]}
KERNEL = 3
HEAD_UNITS = 64
DROPOUT = 0.3


def conv_block_tf(x, filters):
    x = layers.Conv1D(filters, KERNEL, padding="same")(x)
    x = layers.BatchNormalization(momentum=0.9, epsilon=1e-5)(x)
    return layers.ReLU()(x)


def build_tf(model_name, n_features):
    inputs = keras.Input(shape=(n_features, 1))
    x = inputs
    for filters in ARCH[model_name]:
        x = conv_block_tf(x, filters)
    x = layers.Flatten()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(HEAD_UNITS, activation="relu")(x)
    outputs = layers.Dense(1)(x)                    # logit
    return keras.Model(inputs, outputs, name=f"{DATASET}_TF_{model_name.replace('-', '')}")

In [ ]:
patient = X_train_tf[:1]
for m in MODELS:
    model = build_tf(m, N_FEATURES)
    x, rows = patient, []
    for layer in model.layers[1:]:
        x = layer(x, training=False)
        rows.append((layer.__class__.__name__, tuple(x.shape[1:]), layer.count_params()))
    print(f"{m}: one patient through every layer")
    display(pd.DataFrame(rows, columns=["layer", "output shape", "params"]).T)

### 4.4 Parameters

| Layer | Formula | Example |
|---|---|---|
| Conv1D | `(K·C_in + 1)·F` | first layer `(3·1 + 1)·32 = 128` |
| BatchNorm | `2·F` (γ, β) | `2·32 = 64` |
| Dense after Flatten | `(d·F_last + 1)·64` | most of the parameters sit here |
| output | `(64 + 1)·1` | 65 |

Only trainable parameters are counted. The PyTorch models must match (section 5).

In [ ]:
def expected_params(model_name, n_features):
    rows, c = [], 1
    for i, f in enumerate(ARCH[model_name], start=1):
        rows.append((f"conv{i}", f"({KERNEL}·{c} + 1)·{f}", (KERNEL * c + 1) * f))
        rows.append((f"bn{i}", f"2·{f}", 2 * f))
        c = f
    rows.append(("dense", f"({n_features}·{c} + 1)·{HEAD_UNITS}", (n_features * c + 1) * HEAD_UNITS))
    rows.append(("output", f"({HEAD_UNITS} + 1)·1", HEAD_UNITS + 1))
    return pd.DataFrame(rows, columns=["layer", "formula", "params"])


def trainable_params_tf(model):
    return int(sum(np.prod(w.shape) for w in model.trainable_weights))


for m in MODELS:
    table = expected_params(m, N_FEATURES)
    print(f"{m}: {table.params.sum():,} trainable parameters")
    display(table.set_index("layer").T)

### 4.5 Loss and training

- **Sigmoid:** `p = 1 / (1 + e^(−z))`. `z = 0 → p = 0.5`, `z = 1.2 → p = 0.77`, `z = −2 → p = 0.12`.
- **Binary cross-entropy:** `L = −[y·ln p + (1 − y)·ln(1 − p)]`, times the row weight.
  - `p = 0.77`, true `y = 1` → `L = −ln 0.77 = 0.26` (good guess, small loss)
  - `p = 0.77`, true `y = 0` → `L = −ln 0.23 = 1.47` (wrong, large loss)
  - `from_logits=True`: Keras computes the sigmoid inside the loss (numerically safer).
- **Accuracy during training:** `z > 0` (same as `p > 0.5`).
- **Adam**, learning rate 0.001, batch 256 (≈ 820 updates per epoch on 210k rows).
- **Early stopping:** stop after 3 epochs without a lower validation loss, keep the best weights; max 20 epochs.
- **Time per epoch** = median of epochs 2+ (epoch 1 includes setup).

In [ ]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._start)


def train_tf(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, epsilon=1e-7),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.0)],
    )
    timer = EpochTimer()
    stopper = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[stopper, timer], verbose=2)
    hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    return hist, timer.times, int(np.argmin(hist["val_loss"])) + 1

### 4.6 Test metrics

Prediction: `ŷ = 1` if `p > 0.5`.

| Metric | Meaning |
|---|---|
| accuracy | correct / all |
| precision (class 1) | of patients predicted diabetic, the share who are |
| recall (class 1) | of diabetic patients, the share found |
| F1 | `2·P·R / (P + R)`; `f1_positive` = class 1, `f1_macro` = mean of both classes |
| ROC-AUC | chance that a random diabetic patient gets a higher `p` than a random non-diabetic one (0.5 = guessing, 1 = perfect); no threshold needed |
| average precision | area under the precision-recall curve |

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def evaluate_tf(model):
    model.predict(test_ds.take(1), verbose=0)           # warm-up, not timed
    with Stopwatch() as t:
        logits = model.predict(test_ds, verbose=0).ravel()
    probs = sigmoid(logits)
    y_pred = (probs > 0.5).astype(np.int64)
    return y_pred, probs, classification_metrics(y_test, y_pred, probs, binary=True), 1000 * t.seconds / len(y_test)


def make_record(run_id, framework, model_name, params, hist, epoch_times, best_epoch, metrics, ms_per_1k):
    return {
        "run_id": run_id, "dataset": DATASET, "framework": framework, "model": model_name,
        "params": params, "epochs_run": len(epoch_times), "best_epoch": best_epoch,
        "train_time_s": float(sum(epoch_times)),
        "time_per_epoch_s": float(np.median(epoch_times[1:] if len(epoch_times) > 1 else epoch_times)),
        "inference_ms_per_1k": float(ms_per_1k), "test_metrics": metrics,
        "history": {k: hist[k] for k in ("loss", "val_loss", "accuracy", "val_accuracy")},
        "class_weights": class_weight.tolist(), "smoke": SMOKE,
    }


def run_tf(model_name, builder=None, check_params=True):
    run_id = f"{DATASET}-TF-{model_name}"
    keras.utils.set_random_seed(SEED)
    model = (builder or build_tf)(model_name, N_FEATURES)
    params = trainable_params_tf(model)
    if check_params:
        assert params == expected_params(model_name, N_FEATURES).params.sum()
    model.summary()
    hist, epoch_times, best_epoch = train_tf(model)
    y_pred, probs, metrics, ms_per_1k = evaluate_tf(model)
    record = make_record(run_id, "TF", model_name, params, hist, epoch_times, best_epoch, metrics, ms_per_1k)
    save_result(record)
    model.save(MODELS_DIR / f"{run_id}.keras")
    print(pd.Series(metrics).round(4).to_string())
    plot_history(record["history"], run_id)
    plot_confusion(y_test, y_pred, CLASS_LABELS, run_id)
    return model, probs, record

### 4.7 `DB-TF-CNN-3`
Build → check parameters → train → test → save `results/DB-TF-CNN-3.json` → curves and confusion matrix.

In [ ]:
tf_models, tf_probs = {}, {}
tf_models["CNN-3"], tf_probs["CNN-3"], _ = run_tf("CNN-3")

### 4.8 `DB-TF-CNN-5`

In [ ]:
tf_models["CNN-5"], tf_probs["CNN-5"], _ = run_tf("CNN-5")

### 4.9 ROC and precision-recall curves

Both curves move the threshold from 1 down to 0:
- **ROC:** true-positive rate (recall) vs false-positive rate. The diagonal = guessing.
- **Precision-recall:** how precision drops as more diabetic patients are found. Dotted line = share of diabetic rows.

Second figure: predicted `p` for truly diabetic vs non-diabetic test patients; dashed line = threshold 0.5.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve


def plot_threshold_curves(probs_by_model, framework):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for m, p in probs_by_model.items():
        fpr, tpr, _ = roc_curve(y_test, p)
        prec, rec, _ = precision_recall_curve(y_test, p)
        axes[0].plot(fpr, tpr, label=f"{DATASET}-{framework}-{m}")
        axes[1].plot(rec, prec, label=f"{DATASET}-{framework}-{m}")
    axes[0].plot([0, 1], [0, 1], "k:", lw=1)
    axes[0].set(xlabel="false-positive rate", ylabel="true-positive rate", title="ROC curve")
    axes[1].axhline(y_test.mean(), color="k", ls=":", lw=1)
    axes[1].set(xlabel="recall", ylabel="precision", title="Precision-recall curve")
    for ax in axes:
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); savefig(fig, f"{DATASET}-{framework}_roc_pr"); plt.show()

    run_id = f"{DATASET}-{framework}-CNN-5"
    fig, ax = plt.subplots(figsize=(6, 3.8))
    for cls, color in [(0, "tab:blue"), (1, "tab:orange")]:
        ax.hist(probs_by_model["CNN-5"][y_test == cls], bins=50, alpha=0.5, density=True, color=color, label=CLASS_LABELS[cls])
    ax.axvline(0.5, color="k", ls="--", lw=1)
    ax.set(xlabel="predicted probability of diabetes", title=f"{run_id}: p by true class")
    ax.legend(fontsize=8)
    fig.tight_layout(); savefig(fig, f"{run_id}_probabilities"); plt.show()


plot_threshold_curves(tf_probs, "TF")

## 5. PyTorch

Same models, rows, weights and settings:

| Step | Keras | PyTorch |
|---|---|---|
| data | `tf.data` | `TensorDataset` + `DataLoader` (rows, labels, weights) |
| layout | `(N, d, 1)` channels last | `(N, 1, d)` channels first |
| conv | `Conv1D(32, 3, padding="same")` | `Conv1d(1, 32, 3, padding=1)` (input channels explicit) |
| after Flatten | `Dense(64)` | `Linear(64·d, 64)` (input size explicit) |
| loss | `BinaryCrossentropy(from_logits=True)` + weights | `BCEWithLogitsLoss(reduction="none")`, × weights, mean |
| initial weights | Glorot uniform | `xavier_uniform_` (= Glorot), set by hand |
| BatchNorm momentum | 0.9 (old value) | 0.1 (new value): same update |
| training | `fit` | hand-written loop |

Flatten order differs (Keras flattens `(d, F)`, PyTorch `(F, d)`): same numbers in another order, same parameter count.

### 5.1 Data loader

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_loader(x, y, w=None, training=False):
    tensors = [torch.from_numpy(x), torch.from_numpy(y.astype(np.float32)),
               torch.from_numpy(w if w is not None else np.ones(len(y), dtype=np.float32))]
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(*tensors), batch_size=BATCH_SIZE, shuffle=training,
                      generator=g if training else None)


train_loader = make_loader(X_train_pt, y_train, w_train, training=True)
val_loader = make_loader(X_val_pt, y_val)
test_loader = make_loader(X_test_pt, y_test)
xb, yb, wb = next(iter(train_loader))
print("batch:", tuple(xb.shape), "| labels:", tuple(yb.shape), "| weights:", tuple(wb.shape))

### 5.2 Model class
`__init__` creates the layers, `forward` runs them; `squeeze(1)` turns `(256, 1)` into `(256,)`. The check confirms the Keras parameter count.

In [ ]:
def init_like_keras(module):
    if isinstance(module, (nn.Conv1d, nn.Linear)):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)


class CNN1D(nn.Module):
    def __init__(self, model_name, n_features):
        super().__init__()
        blocks, c = [], 1
        for filters in ARCH[model_name]:
            blocks += [nn.Conv1d(c, filters, kernel_size=KERNEL, padding=KERNEL // 2),
                       nn.BatchNorm1d(filters, momentum=0.1, eps=1e-5),
                       nn.ReLU()]
            c = filters
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(DROPOUT),
                                  nn.Linear(c * n_features, HEAD_UNITS), nn.ReLU(),
                                  nn.Linear(HEAD_UNITS, 1))          # logit
        self.apply(init_like_keras)

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)


def trainable_params_pt(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for m in MODELS:
    n_pt = trainable_params_pt(CNN1D(m, N_FEATURES))
    n_formula = expected_params(m, N_FEATURES).params.sum()
    assert n_pt == n_formula, (m, n_pt, n_formula)
    print(f"{m}: PyTorch {n_pt:,} = formula {n_formula:,} = Keras (match)")

### 5.3 Training loop

Per batch: `zero_grad()` → `logits = model(x)` → `loss = mean(BCE(logits, y) × w)` (same formula as 4.5)
→ `backward()` → `step()`. `model.train()` / `model.eval()` switch dropout and BatchNorm mode.
Validation loss is unweighted, as in Keras. Early stopping: 3 epochs without improvement → stop, reload best weights.

In [ ]:
def sync():
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    elif DEVICE.type == "cuda":
        torch.cuda.synchronize()


bce = nn.BCEWithLogitsLoss(reduction="none")


@torch.no_grad()
def evaluate_loader(model, loader):
    model.eval()
    total_loss, correct, n, logits_all = 0.0, 0, 0, []
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += bce(logits, y).sum().item()
        correct += ((logits > 0).float() == y).sum().item()
        n += len(y)
        logits_all.append(logits.cpu())
    return total_loss / n, correct / n, torch.cat(logits_all)


def train_pt(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, eps=1e-7)
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    epoch_times, best_loss, best_state, wait = [], float("inf"), None, 0

    for epoch in range(1, EPOCHS + 1):
        sync(); start = time.perf_counter()
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for x, y, w in train_loader:
            x, y, w = x.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = (bce(logits, y) * w).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            correct += ((logits > 0).float() == y).sum().item()
            n += len(y)
        val_loss, val_acc, _ = evaluate_loader(model, val_loader)
        sync(); epoch_times.append(time.perf_counter() - start)

        for k, v in zip(hist, (total_loss / n, correct / n, val_loss, val_acc)):
            hist[k].append(float(v))
        print(f"Epoch {epoch}/{EPOCHS} - {epoch_times[-1]:.1f}s - loss: {hist['loss'][-1]:.4f} - "
              f"accuracy: {hist['accuracy'][-1]:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

        if val_loss < best_loss:
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Epoch {epoch}: early stopping")
                break

    model.load_state_dict(best_state)
    best_epoch = int(np.argmin(hist["val_loss"])) + 1
    print(f"Restoring model weights from the end of the best epoch: {best_epoch}.")
    return hist, epoch_times, best_epoch


def evaluate_pt(model):
    evaluate_loader(model, [next(iter(test_loader))])                  # warm-up
    sync()
    with Stopwatch() as t:
        _, _, logits = evaluate_loader(model, test_loader)
        sync()
    probs = torch.sigmoid(logits).numpy()
    y_pred = (probs > 0.5).astype(np.int64)
    return y_pred, probs, classification_metrics(y_test, y_pred, probs, binary=True), 1000 * t.seconds / len(y_test)


def run_pt(model_name):
    run_id = f"{DATASET}-PT-{model_name}"
    torch.manual_seed(SEED)
    model = CNN1D(model_name, N_FEATURES).to(DEVICE)
    params = trainable_params_pt(model)
    print(model)
    print(f"trainable parameters: {params:,}")
    hist, epoch_times, best_epoch = train_pt(model)
    y_pred, probs, metrics, ms_per_1k = evaluate_pt(model)
    record = make_record(run_id, "PT", model_name, params, hist, epoch_times, best_epoch, metrics, ms_per_1k)
    save_result(record)
    torch.save(model.state_dict(), MODELS_DIR / f"{run_id}.pt")
    print(pd.Series(metrics).round(4).to_string())
    plot_history(record["history"], run_id)
    plot_confusion(y_test, y_pred, CLASS_LABELS, run_id)
    return model, probs, record

### 5.4 `DB-PT-CNN-3`

In [ ]:
pt_models, pt_probs = {}, {}
pt_models["CNN-3"], pt_probs["CNN-3"], _ = run_pt("CNN-3")

### 5.5 `DB-PT-CNN-5`

In [ ]:
pt_models["CNN-5"], pt_probs["CNN-5"], _ = run_pt("CNN-5")

### 5.6 ROC and precision-recall curves

In [ ]:
plot_threshold_curves(pt_probs, "PT")

## 6. Comparison

### 6.1 MLP baseline (`DB-TF-MLP`)

`Dense 64 → ReLU → Dense 64 → ReLU → Dense 1` on the plain feature vector. Every feature connects to every unit,
so column order does not matter. If it matches the CNNs, the "neighbouring features" idea of Conv1D is not
helping on this table. Same data, weights and training settings; not part of the CNN-3 vs CNN-5 comparison.

In [ ]:
def build_mlp(model_name, n_features):
    inputs = keras.Input(shape=(n_features, 1))
    x = layers.Flatten()(inputs)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    return keras.Model(inputs, layers.Dense(1)(x), name=f"{DATASET}_TF_MLP")


_, mlp_probs, _ = run_tf("MLP", builder=build_mlp, check_params=False)

### 6.2 Results

Read from the `results/<run_id>.json` files (the same files the report uses).
`time_per_epoch_s` = median of epochs 2+; `inference_ms_per_1k` = time to predict 1,000 patients.

In [ ]:
records = load_results(RUN_IDS + [BASELINE_ID])
comparison = results_table(records)
comparison.to_csv(RESULTS_DIR / f"{DATASET}_comparison.csv")
comparison.round(4)

Charts (two per figure) and the differences **CNN-5 − CNN-3** and **PT − TF**.

In [ ]:
quality = ["accuracy", "f1_macro", "f1_positive", "roc_auc"]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
comparison[quality].plot.bar(ax=axes[0], rot=0)
axes[0].set_ylim(max(0, comparison[quality].min().min() - 0.1), 1)
axes[0].set_title(f"{DATASET}: test metrics"); axes[0].grid(axis="y", alpha=0.3); axes[0].legend(fontsize=8)
comparison["time_per_epoch_s"].plot.bar(ax=axes[1], rot=0, color="gray")
axes[1].set_title(f"{DATASET}: training time per epoch (s)"); axes[1].grid(axis="y", alpha=0.3)
for ax in axes:
    ax.tick_params(axis="x", labelsize=8)
fig.tight_layout(); savefig(fig, f"{DATASET}_comparison_bars"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for r in records:
    if r["run_id"] == BASELINE_ID:
        continue
    style = "-" if r["framework"] == "TF" else "--"
    ep = np.arange(1, len(r["history"]["val_loss"]) + 1)
    axes[0].plot(ep, r["history"]["val_loss"], style, marker="o", label=r["run_id"])
    axes[1].plot(ep, r["history"]["val_accuracy"], style, marker="o", label=r["run_id"])
axes[0].set_title(f"{DATASET}: validation loss"); axes[1].set_title(f"{DATASET}: validation accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, f"{DATASET}_comparison_curves"); plt.show()

cols = ["accuracy", "f1_macro", "roc_auc", "params", "time_per_epoch_s", "epochs"]
by = comparison[comparison.model.isin(MODELS)].set_index(["framework", "model"])[cols]
print("CNN-5 minus CNN-3 (per framework):")
display(pd.DataFrame({fw: by.loc[(fw, "CNN-5")] - by.loc[(fw, "CNN-3")] for fw in FRAMEWORKS}).T.round(4))
print("PT minus TF (per model):")
display(pd.DataFrame({m: by.loc[("PT", m)] - by.loc[("TF", m)] for m in MODELS}).T.round(4))

**Discussion** *(write after the full run)*
- Does CNN-5 beat CNN-3 on ROC-AUC and macro F1? Worth the extra parameters?
- Do TF and PT agree? Which is faster?
- How close is the MLP baseline, and what does that say about convolutions on tables?
- Would a threshold other than 0.5 suit this task (probability histogram)?

## 7. Conclusion

*(3–5 takeaways, written after the full run)*
1. …
2. …
3. …